# Meeting Audio Pipeline — Kaggle GPU

**Stages:** normalize → denoise (DeepFilterNet) → [speaker separation] → VAD (Silero) → ASR (Whisper / Qwen3-ASR) → [WER evaluation]

**GPU:** Enable via *Settings → Accelerator → GPU T4 x2 or P100* before running.

---
| Cell | What it does |
|---|---|
| 1 | Install dependencies |
| 2 | Configuration — edit before running |
| 3 | Source code (all modules inlined) |
| 4 | Stage 1 — normalize → denoise → VAD |
| 5 | Stage 2 — ASR |
| 6 | Stage 3 (optional) — WER evaluation |
| 7 | View results |

## 1 · Install dependencies

In [ ]:
# PyTorch with CUDA is pre-installed on Kaggle — no need to reinstall.
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

# DeepFilterNet: install without deps to skip numpy<2 constraint
pip("DeepFilterNet==0.5.6", "DeepFilterLib==0.5.6", "--no-deps")

pip(
    "clearvoice==0.1.2",
    "faster-whisper>=1.0.0",
    "transformers>=4.51.0",
    "accelerate>=0.30.0",
    "soundfile>=0.12.1",
    "librosa>=0.10.1",
    "huggingface_hub>=0.24.0",
    "datasets>=2.0.0",
    "jiwer>=3.0.0",
)

print("All packages installed.")

## 2 · Configuration

In [ ]:
# ── Input / output ────────────────────────────────────────────────────────────
INPUT_FILE  = "/kaggle/input/your-dataset/your_audio.mp4"  # any format ffmpeg supports
OUTPUT_DIR  = "/kaggle/working/output"

# ── ASR backend ───────────────────────────────────────────────────────────────
# "whisper" : faster-whisper  — tiny | base | small | medium | large-v1 | large-v2 | large-v3
# "qwen"    : Qwen3-ASR       — Qwen/Qwen3-ASR-0.6B | Qwen/Qwen3-ASR-1.7B
ASR_BACKEND = "whisper"
ASR_MODEL   = "large-v3"

# ── Pipeline options ──────────────────────────────────────────────────────────
SEPARATE_SPEAKERS  = False   # True → ClearVoice MossFormer2 speech separation before VAD
CHUNK_SECONDS      = 30      # DeepFilterNet processing chunk size
MAX_CHUNK_DURATION = 25.0    # max merged ASR chunk length (seconds)
MAX_GAP            = 1.5     # max silence gap to merge into one chunk (seconds)

# ── WER evaluation (optional) ─────────────────────────────────────────────────
# Set to a .json or .csv file with ground-truth transcripts, or leave None to skip.
# JSON: list of {"file": "name.wav", "text": "..."}
# CSV:  columns file,text  (or audio,transcript — auto-detected)
METADATA_PATH = None   # e.g. "/kaggle/input/your-dataset/metadata.json"

## 3 · Source code

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# src/utils/file_utils.py
# ─────────────────────────────────────────────────────────────────────────────
import os

def ensure_dir(path):
    os.makedirs(path, exist_ok=True)

def temp_path(directory, filename):
    ensure_dir(directory)
    return os.path.join(directory, filename)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# src/utils/audio_utils.py
# ─────────────────────────────────────────────────────────────────────────────
from __future__ import annotations
import tempfile
import torch
import torchaudio

def extract_chunk(
    audio_path: str,
    start: float,
    end: float,
    target_sr: int = 16_000,
) -> str:
    """
    Extract a time slice from audio_path, resample to target_sr,
    and write to a temporary WAV file.
    Returns the temp file path (caller is responsible for deletion).
    """
    wav, sr = torchaudio.load(audio_path)
    if sr != target_sr:
        wav = torchaudio.functional.resample(wav, sr, target_sr)
    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)
    s = int(start * target_sr)
    e = int(end * target_sr)
    chunk = wav[:, s:e]
    tmp = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
    torchaudio.save(tmp.name, chunk, target_sr)
    tmp.close()
    return tmp.name

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# src/preprocessing/normalizer.py
# ─────────────────────────────────────────────────────────────────────────────
import subprocess

def _run_cmd(cmd):
    result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    if result.returncode != 0:
        raise RuntimeError(
            f"Command failed:\n{' '.join(cmd)}\n{result.stderr.decode()}"
        )
    return result

def normalize(input_file: str, output_wav: str):
    """Convert input audio to mono, 48 kHz, loudnorm WAV via ffmpeg."""
    _run_cmd([
        "ffmpeg", "-y",
        "-i", input_file,
        "-ac", "1",
        "-ar", "48000",
        "-af", "loudnorm",
        "-c:a", "pcm_s16le",
        output_wav,
    ])

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# src/enhancement/noise_reducer.py
# ─────────────────────────────────────────────────────────────────────────────
import math
import os
import torch
import torchaudio
from df.enhance import enhance, init_df

class NoiseReducer:
    def __init__(self, chunk_seconds=30):
        print("Loading DeepFilterNet model...")
        self.model, self.df_state, _ = init_df()
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        self.chunk_seconds = chunk_seconds
        print(f"Using device: {self.device}")
        print(f"Chunk size: {self.chunk_seconds} seconds")

    def process(self, input_wav: str, output_wav: str):
        """Apply DeepFilterNet noise reduction in chunks, then save result."""
        print("Loading preprocessed audio...")
        audio, sr = torchaudio.load(input_wav)
        print(f"Sample rate: {sr}")
        print(f"Audio shape: {audio.shape}")

        chunk_samples = sr * self.chunk_seconds
        total_samples = audio.shape[1]
        num_chunks = math.ceil(total_samples / chunk_samples)
        print(f"Total chunks: {num_chunks}")

        enhanced_chunks = []
        for i in range(num_chunks):
            start = i * chunk_samples
            end   = min((i + 1) * chunk_samples, total_samples)
            print(f"Processing chunk {i+1}/{num_chunks} ({start}:{end})")
            chunk = audio[:, start:end].to(self.device)
            with torch.no_grad():
                enhanced_chunk = enhance(self.model, self.df_state, chunk)
            enhanced_chunks.append(enhanced_chunk.cpu())
            del chunk, enhanced_chunk
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        print("Concatenating chunks...")
        enhanced_audio = torch.cat(enhanced_chunks, dim=1)

        output_dir = os.path.dirname(output_wav)
        if output_dir:
            os.makedirs(output_dir, exist_ok=True)
        print("Saving denoised audio...")
        torchaudio.save(output_wav, enhanced_audio, sr)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# src/enhancement/speech_separator.py
# ─────────────────────────────────────────────────────────────────────────────
import os
import torch
import torchaudio
from clearvoice import ClearVoice

class SpeechSeparator:
    # MossFormer2_SS_16K requires 16kHz mono audio
    TARGET_SR  = 16000
    MODEL_NAME = "MossFormer2_SS_16K"

    def __init__(self):
        if not torch.cuda.is_available():
            n = os.cpu_count() or 1
            torch.set_num_threads(n)
            torch.set_num_interop_threads(n)
            print(f"CPU mode: using {n} threads")
        print("Loading MossFormer2_SS_16K model via ClearVoice...")
        self.cv = ClearVoice(task="speech_separation", model_names=[self.MODEL_NAME])
        print("MossFormer2_SS_16K ready.")

    def process(self, input_wav: str, output_dir: str) -> list:
        """
        Separate mixed speech into individual speaker tracks.
        Returns list of output file paths, one per speaker.
        """
        os.makedirs(output_dir, exist_ok=True)

        # resample to 16kHz if needed
        audio, sr = torchaudio.load(input_wav)
        if sr != self.TARGET_SR:
            resampler = torchaudio.transforms.Resample(sr, self.TARGET_SR)
            audio = resampler(audio)
            resampled_path = os.path.join(output_dir, "_input_16k.wav")
            torchaudio.save(resampled_path, audio, self.TARGET_SR)
            input_wav = resampled_path

        self.cv(input_path=input_wav, online_write=True, output_path=output_dir)

        # ClearVoice writes to output_dir/<MODEL_NAME>/
        model_out_dir = os.path.join(output_dir, self.MODEL_NAME)
        search_dir = model_out_dir if os.path.isdir(model_out_dir) else output_dir

        output_files = sorted([
            os.path.join(search_dir, f)
            for f in os.listdir(search_dir)
            if f.endswith(".wav") and not f.startswith("_input")
        ])

        # clean up temp resample file if created
        tmp = os.path.join(output_dir, "_input_16k.wav")
        if os.path.exists(tmp):
            os.remove(tmp)

        print(f"Separated into {len(output_files)} speaker track(s).")
        return output_files

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# src/vad/segmenter.py
# ─────────────────────────────────────────────────────────────────────────────
from __future__ import annotations
from dataclasses import dataclass
import torch
import torchaudio

@dataclass
class AudioChunk:
    start: float   # seconds
    end:   float   # seconds

    @property
    def duration(self) -> float:
        return self.end - self.start

    def __repr__(self) -> str:
        return f"AudioChunk({self.start:.2f}s–{self.end:.2f}s, {self.duration:.2f}s)"


class VADSegmenter:
    """
    Wraps Silero VAD to produce silence-aware ASR chunks.
    VAD finds raw speech segments; merge step groups adjacent segments
    into longer chunks so ASR models get enough context.
    """
    TARGET_SR = 16_000  # Silero VAD requires 16 kHz

    def __init__(
        self,
        min_speech_ms: int = 300,
        min_silence_ms: int = 500,
        max_chunk_duration: float = 25.0,
        max_gap: float = 1.5,
        min_chunk_duration: float = 2.0,
    ):
        self.min_speech_ms      = min_speech_ms
        self.min_silence_ms     = min_silence_ms
        self.max_chunk_duration = max_chunk_duration
        self.max_gap            = max_gap
        self.min_chunk_duration = min_chunk_duration

        print("Loading Silero VAD...")
        self.model, self._utils = torch.hub.load(
            repo_or_dir="snakers4/silero-vad",
            model="silero_vad",
            trust_repo=True,
        )
        self._get_ts    = self._utils[0]   # get_speech_timestamps
        self._read_audio = self._utils[2]  # read_audio
        print("Silero VAD ready.")

    def get_chunks(self, audio_path: str) -> list[AudioChunk]:
        """Full pipeline: VAD on audio file → merge into ASR-ready chunks."""
        raw_segments = self._run_vad(audio_path)
        chunks = self._merge(raw_segments)
        print(
            f"VAD: {len(raw_segments)} speech segment(s) "
            f"→ {len(chunks)} ASR chunk(s)."
        )
        return chunks

    def _run_vad(self, audio_path: str) -> list[AudioChunk]:
        wav, sr = torchaudio.load(audio_path)
        if sr != self.TARGET_SR:
            wav = torchaudio.functional.resample(wav, sr, self.TARGET_SR)
        if wav.shape[0] > 1:
            wav = wav.mean(dim=0)
        timestamps = self._get_ts(
            wav,
            self.model,
            sampling_rate=self.TARGET_SR,
            min_speech_duration_ms=self.min_speech_ms,
            min_silence_duration_ms=self.min_silence_ms,
            return_seconds=True,
        )
        return [AudioChunk(start=t["start"], end=t["end"]) for t in timestamps]

    def _merge(self, segments: list[AudioChunk]) -> list[AudioChunk]:
        """
        Merge adjacent segments into longer chunks, respecting:
        - gap between consecutive segments ≤ max_gap
        - resulting chunk duration ≤ max_chunk_duration
        A second pass absorbs chunks shorter than min_chunk_duration
        into their nearest neighbour.
        """
        if not segments:
            return []

        chunks: list[AudioChunk] = []
        c_start = segments[0].start
        c_end   = segments[0].end

        for seg in segments[1:]:
            gap = seg.start - c_end
            would_be_duration = seg.end - c_start
            if gap <= self.max_gap and would_be_duration <= self.max_chunk_duration:
                c_end = seg.end
            else:
                chunks.append(AudioChunk(start=c_start, end=c_end))
                c_start, c_end = seg.start, seg.end
        chunks.append(AudioChunk(start=c_start, end=c_end))

        # Second pass: absorb short chunks into nearest neighbour
        merged: list[AudioChunk] = []
        i = 0
        while i < len(chunks):
            chunk = chunks[i]
            if chunk.duration < self.min_chunk_duration:
                gap_prev = (chunk.start - merged[-1].end) if merged else float("inf")
                gap_next = (chunks[i+1].start - chunk.end) if i+1 < len(chunks) else float("inf")
                if gap_prev <= gap_next and merged:
                    merged[-1] = AudioChunk(start=merged[-1].start, end=chunk.end)
                elif i+1 < len(chunks):
                    chunks[i+1] = AudioChunk(start=chunk.start, end=chunks[i+1].end)
                else:
                    merged.append(chunk)
            else:
                merged.append(chunk)
            i += 1

        return merged

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# src/transcription/base.py
# ─────────────────────────────────────────────────────────────────────────────
from __future__ import annotations
from abc import ABC, abstractmethod
from dataclasses import dataclass, field

@dataclass
class TranscribedSegment:
    start:    float
    end:      float
    text:     str
    language: str = ""
    speaker:  str = ""              # empty when diarization is not used
    words:    list[dict] = field(default_factory=list)  # [{word, start, end}]

    def __repr__(self) -> str:
        tag = f" ({self.language})" if self.language else ""
        spk = f"{self.speaker}: "  if self.speaker  else ""
        return f"[{self.start:.2f}s–{self.end:.2f}s]{tag} {spk}{self.text}"


class BaseTranscriber(ABC):
    """Common interface for all ASR backends."""

    @abstractmethod
    def transcribe_chunk(
        self,
        audio_path: str,
        start: float,
        end: float,
        language: str | None = None,
    ) -> TranscribedSegment:
        """
        Transcribe a time slice of audio_path.
        start, end: boundaries in seconds.
        language: BCP-47 hint; None = auto-detect.
        """
        ...

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# src/transcription/whisper_transcriber.py
# ─────────────────────────────────────────────────────────────────────────────
from __future__ import annotations
import io
import numpy as np
import soundfile as sf
import torch
import torchaudio
from faster_whisper import WhisperModel

_WHISPER_SR = 16_000

WHISPER_MODELS: list[tuple[str, str]] = [
    ("tiny (39M)",      "tiny"),
    ("base (74M)",      "base"),
    ("small (244M)",    "small"),
    ("medium (769M)",   "medium"),
    ("large-v1 (1.5B)", "large-v1"),
    ("large-v2 (1.5B)", "large-v2"),
    ("large-v3 (1.5B)", "large-v3"),
]

class WhisperTranscriber(BaseTranscriber):
    """Multilingual Whisper ASR; model_name selects the variant."""

    def __init__(self, model_name: str = "large-v3"):
        device  = "cuda" if torch.cuda.is_available() else "cpu"
        compute = "float16" if device == "cuda" else "int8"
        print(f"Loading Whisper '{model_name}' on {device} ({compute})...")
        self.model = WhisperModel(model_name, device=device, compute_type=compute)
        print(f"WhisperTranscriber ({model_name}) ready.")

    def transcribe_chunk(
        self,
        audio_path: str,
        start: float,
        end: float,
        language: str | None = None,
    ) -> TranscribedSegment:
        buf = self._load_buffer(audio_path, start, end)
        if buf is None:
            return TranscribedSegment(start=start, end=end, text="", language="")
        segments, info = self.model.transcribe(
            buf, language=language, beam_size=5, vad_filter=False
        )
        text = " ".join(s.text.strip() for s in segments)
        detected_lang = info.language or (language or "")
        return TranscribedSegment(start=start, end=end, text=text, language=detected_lang)

    def _load_buffer(self, audio_path: str, start: float, end: float) -> io.BytesIO | None:
        wav, sr = torchaudio.load(audio_path)
        if sr != _WHISPER_SR:
            wav = torchaudio.functional.resample(wav, sr, _WHISPER_SR)
        if wav.shape[0] > 1:
            wav = wav.mean(dim=0, keepdim=True)
        s = int(start * _WHISPER_SR)
        e = int(end   * _WHISPER_SR)
        chunk = wav[0, s:e]
        if chunk.numel() == 0:
            return None
        buf = io.BytesIO()
        sf.write(buf, chunk.numpy().astype(np.float32), _WHISPER_SR, format="WAV")
        buf.seek(0)
        return buf

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# src/transcription/qwen_transcriber.py
# ─────────────────────────────────────────────────────────────────────────────
from __future__ import annotations
import numpy as np
import torch
import torchaudio

_QWEN_SR = 16_000

QWEN_MODELS: list[tuple[str, str]] = [
    ("Qwen3-ASR 0.6B", "Qwen/Qwen3-ASR-0.6B"),
    ("Qwen3-ASR 1.7B", "Qwen/Qwen3-ASR-1.7B"),
]

class QwenTranscriber(BaseTranscriber):
    """Qwen3-ASR multilingual speech-to-text via transformers."""

    def __init__(self, model_id: str = "Qwen/Qwen3-ASR-0.6B"):
        from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor
        device = "cuda" if torch.cuda.is_available() else "cpu"
        dtype  = torch.float16 if device == "cuda" else torch.float32
        print(f"Loading {model_id} on {device}...")
        self.processor = AutoProcessor.from_pretrained(model_id)
        self.model = AutoModelForSpeechSeq2Seq.from_pretrained(
            model_id, torch_dtype=dtype, device_map="auto"
        )
        self.dtype = dtype
        print(f"QwenTranscriber ({model_id}) ready.")

    def transcribe_chunk(
        self,
        audio_path: str,
        start: float,
        end: float,
        language: str | None = None,
    ) -> TranscribedSegment:
        audio_np = self._load(audio_path, start, end)
        if audio_np is None:
            return TranscribedSegment(start=start, end=end, text="")
        inputs = self.processor(
            audio_np, sampling_rate=_QWEN_SR,
            return_tensors="pt", return_attention_mask=True,
        )
        inputs = {k: v.to(self.model.device) for k, v in inputs.items()}
        if self.dtype == torch.float16 and "input_features" in inputs:
            inputs["input_features"] = inputs["input_features"].half()
        with torch.no_grad():
            ids = self.model.generate(**inputs, language=None, task="transcribe")
        text = self.processor.batch_decode(ids, skip_special_tokens=True)[0].strip()
        return TranscribedSegment(start=start, end=end, text=text)

    def _load(self, audio_path: str, start: float, end: float) -> np.ndarray | None:
        wav, sr = torchaudio.load(audio_path)
        if sr != _QWEN_SR:
            wav = torchaudio.functional.resample(wav, sr, _QWEN_SR)
        if wav.shape[0] > 1:
            wav = wav.mean(dim=0, keepdim=True)
        chunk = wav[0, int(start * _QWEN_SR):int(end * _QWEN_SR)]
        return chunk.numpy().astype(np.float32) if chunk.numel() else None

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# src/transcription/router.py
# ASR Runner: dispatch to Whisper or Qwen3-ASR backend.
# No LID — both backends are multilingual and auto-detect language.
# ─────────────────────────────────────────────────────────────────────────────

# Expose model lists for UI / notebook selection
BACKEND_MODELS: dict[str, list[tuple[str, str]]] = {
    "whisper": WHISPER_MODELS,
    "qwen":    QWEN_MODELS,
}

class ASRRunner:
    """
    Runs ASR on each VAD chunk using the selected multilingual backend.
    Backends:
        whisper — faster-whisper (tiny/base/small/medium/large-v1/v2/v3)
        qwen    — Qwen3-ASR (0.6B / 1.7B)
    """

    def __init__(self, backend: str = "whisper", model: str = "large-v3"):
        backend = backend.lower()
        if backend == "whisper":
            self._transcriber: BaseTranscriber = WhisperTranscriber(model)
        elif backend == "qwen":
            self._transcriber = QwenTranscriber(model)
        else:
            raise ValueError(
                f"Unknown ASR backend: {backend!r}. Choose 'whisper' or 'qwen'."
            )

    def transcribe(
        self,
        audio_path: str,
        chunks: list[AudioChunk],
    ) -> list[TranscribedSegment]:
        results: list[TranscribedSegment] = []
        for i, chunk in enumerate(chunks):
            print(f"  [{i+1}/{len(chunks)}] {chunk.start:.1f}s–{chunk.end:.1f}s")
            seg = self._transcriber.transcribe_chunk(audio_path, chunk.start, chunk.end)
            if seg.text:
                results.append(seg)
        results.sort(key=lambda s: s.start)
        return results

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# src/postprocessing/formatter.py
# ─────────────────────────────────────────────────────────────────────────────
from __future__ import annotations
import json
import os

def _srt_timestamp(seconds: float) -> str:
    h  = int(seconds // 3600)
    m  = int((seconds % 3600) // 60)
    s  = int(seconds % 60)
    ms = int((seconds % 1) * 1000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

class Formatter:
    def __init__(self, segments: list[TranscribedSegment]):
        self.segments = sorted(segments, key=lambda s: s.start)

    def to_text(self) -> str:
        """Plain text with speaker labels and timestamps."""
        lines = []
        for seg in self.segments:
            lines.append(
                f"[{seg.start:.2f}s – {seg.end:.2f}s] {seg.speaker}: {seg.text}"
            )
        return "\n".join(lines)

    def to_plain_transcript(self) -> str:
        """
        Merged transcript grouping consecutive turns by the same speaker.
        Best format to pass to a summarizer.
        """
        if not self.segments:
            return ""
        lines = []
        current_speaker = self.segments[0].speaker
        current_texts   = [self.segments[0].text]
        for seg in self.segments[1:]:
            if seg.speaker == current_speaker:
                current_texts.append(seg.text)
            else:
                lines.append(f"{current_speaker}: {' '.join(current_texts)}")
                current_speaker = seg.speaker
                current_texts   = [seg.text]
        lines.append(f"{current_speaker}: {' '.join(current_texts)}")
        return "\n".join(lines)

    def to_srt(self) -> str:
        """SRT subtitle format."""
        blocks = []
        for i, seg in enumerate(self.segments, start=1):
            start = _srt_timestamp(seg.start)
            end   = _srt_timestamp(seg.end)
            blocks.append(f"{i}\n{start} --> {end}\n{seg.speaker}: {seg.text}")
        return "\n\n".join(blocks)

    def to_json(self) -> str:
        """JSON array of all segments."""
        data = [
            {
                "speaker":  seg.speaker,
                "start":    seg.start,
                "end":      seg.end,
                "text":     seg.text,
                "words":    seg.words,
            }
            for seg in self.segments
        ]
        return json.dumps(data, ensure_ascii=False, indent=2)

    def save(self, output_path: str, fmt: str = "txt") -> str:
        """
        Write transcript to file.
        fmt: "txt" | "plain" | "srt" | "json"
        Returns absolute path of the written file.
        """
        ext = os.path.splitext(output_path)[1].lstrip(".")
        fmt = ext if ext in {"txt", "srt", "json"} else fmt
        renderers = {
            "txt":   self.to_text,
            "plain": self.to_plain_transcript,
            "srt":   self.to_srt,
            "json":  self.to_json,
        }
        if fmt not in renderers:
            raise ValueError(f"Unsupported format '{fmt}'. Choose: {list(renderers)}")
        content = renderers[fmt]()
        os.makedirs(os.path.dirname(output_path) or ".", exist_ok=True)
        with open(output_path, "w", encoding="utf-8") as f:
            f.write(content)
        print(f"Transcript saved → {output_path}")
        return os.path.abspath(output_path)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# src/postprocessing/wer.py
# ─────────────────────────────────────────────────────────────────────────────
"""
WER (Word Error Rate) computation — load metadata, match ground truth,
compute per-file and aggregate WER stats.
"""
from __future__ import annotations
import csv
import json
from pathlib import Path
import jiwer
import numpy as np

def load_metadata(path: str) -> list[dict]:
    """Load metadata from JSON or CSV. Returns list of dicts."""
    p = Path(path)
    if p.suffix.lower() == ".csv":
        with open(p, encoding="utf-8") as f:
            return list(csv.DictReader(f))
    with open(p, encoding="utf-8") as f:
        return json.load(f)

def _filename_stems(entry: dict) -> list[str]:
    stems = []
    for key in ("file", "audio", "filename", "file_path", "audio_path"):
        val = entry.get(key)
        if val:
            stems.append(Path(str(val)).stem)
    return stems

def _detect_text_field(entry: dict) -> str:
    for key in ("text", "transcript", "answer", "sentence", "ground_truth"):
        if key in entry:
            return key
    for key in entry:
        if "text" in key.lower() or "transcript" in key.lower():
            return key
    raise ValueError(f"Cannot detect ground-truth field in keys: {list(entry.keys())}")

def build_ground_truth_map(metadata: list[dict]) -> dict[str, str]:
    """Build mapping from filename-stem → ground-truth text."""
    if not metadata:
        return {}
    text_field = _detect_text_field(metadata[0])
    mapping: dict[str, str] = {}
    for entry in metadata:
        stems = _filename_stems(entry)
        gt = (entry.get(text_field) or "").strip()
        for stem in stems:
            if stem:
                mapping[stem] = gt
    return mapping

def _match_ref(basename: str, gt_map: dict[str, str]) -> str | None:
    if basename in gt_map:
        return gt_map[basename]
    stripped = basename.lstrip("0")
    for stem, gt in gt_map.items():
        if stem.lstrip("0") == stripped:
            return gt
    for stem, gt in gt_map.items():
        if stem in basename or basename in stem:
            return gt
    return None

def compute_wer_score(reference: str, hypothesis: str) -> float:
    """Compute WER (0.0–1.0)."""
    if not reference.strip():
        return 0.0
    return float(jiwer.wer(reference, hypothesis))

def compute_wer_stats(
    audio_paths: list[str],
    transcriptions: dict[str, str],
    ground_truth_map: dict[str, str],
) -> dict:
    """
    Compute per-file WER and aggregate statistics.
    Returns dict with keys: per_file, mean, std, min, max, count.
    """
    per_file: list[dict] = []
    scores: list[float] = []
    for ap in audio_paths:
        basename = Path(ap).stem
        hyp = transcriptions.get(basename, "")
        ref = _match_ref(basename, ground_truth_map)
        if ref is not None:
            wer_val = compute_wer_score(ref, hyp)
            scores.append(wer_val)
            per_file.append({
                "name": basename, "wer": wer_val,
                "ref_words": len(ref.split()), "hyp_words": len(hyp.split()),
            })
        else:
            per_file.append({
                "name": basename, "wer": None,
                "ref_words": 0, "hyp_words": len(hyp.split()) if hyp else 0,
            })
    if scores:
        arr = np.array(scores)
        return {
            "per_file": per_file,
            "mean": float(np.mean(arr)), "std": float(np.std(arr)),
            "min":  float(np.min(arr)),  "max": float(np.max(arr)),
            "count": len(scores),
        }
    return {"per_file": per_file, "mean": 0.0, "std": 0.0, "min": 0.0, "max": 0.0, "count": 0}

## 4 · Stage 1 — Preprocess (normalize → denoise → VAD)

In [ ]:
import os, shutil, tempfile
import torch

print(f"CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU            : {torch.cuda.get_device_name(0)}")
    print(f"VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
basename = os.path.splitext(os.path.basename(INPUT_FILE))[0]

noise_reducer = NoiseReducer(chunk_seconds=CHUNK_SECONDS)
separator     = SpeechSeparator() if SEPARATE_SPEAKERS else None
segmenter     = VADSegmenter(
    max_chunk_duration=MAX_CHUNK_DURATION,
    max_gap=MAX_GAP,
)

with tempfile.TemporaryDirectory() as tmpdir:
    normalized_wav = os.path.join(tmpdir, "normalized.wav")
    denoised_wav   = os.path.join(tmpdir, "denoised.wav")

    print(f"[pre] Normalizing: {basename}")
    normalize(INPUT_FILE, normalized_wav)

    print(f"[pre] Denoising: {basename}")
    noise_reducer.process(normalized_wav, denoised_wav)

    audio_dir    = os.path.join(OUTPUT_DIR, "audio")
    os.makedirs(audio_dir, exist_ok=True)
    denoised_out = os.path.join(audio_dir, f"{basename}_denoised.wav")
    shutil.copy(denoised_wav, denoised_out)

    asr_input = denoised_wav

    if separator:
        print(f"[pre] Separating speakers: {basename}")
        separated_dir = os.path.join(OUTPUT_DIR, "separated")
        tracks = separator.process(denoised_wav, separated_dir)
        if tracks:
            asr_input = tracks[0]

    print(f"[pre] VAD: {basename}")
    chunks = segmenter.get_chunks(asr_input)

# Serialise for Stage 2 (can re-run ASR without re-preprocessing)
chunks_data = [{"start": c.start, "end": c.end} for c in chunks]

print(f"\nPreprocessing done.")
print(f"  Denoised audio : {denoised_out}")
print(f"  ASR chunks     : {len(chunks_data)}")

## 5 · Stage 2 — ASR

In [ ]:
chunk_objs = [AudioChunk(c["start"], c["end"]) for c in chunks_data]

if not chunk_objs:
    print("No speech detected — nothing to transcribe.")
else:
    print(f"[ASR] Loading model: {ASR_BACKEND}/{ASR_MODEL}")
    runner = ASRRunner(backend=ASR_BACKEND, model=ASR_MODEL)

    print(f"[ASR] Transcribing {len(chunk_objs)} chunk(s): {basename}")
    transcribed = runner.transcribe(denoised_out, chunk_objs)

    formatter = Formatter(transcribed)

    txt_path  = os.path.join(OUTPUT_DIR, f"{basename}_transcript.txt")
    srt_path  = os.path.join(OUTPUT_DIR, f"{basename}_transcript.srt")
    json_path = os.path.join(OUTPUT_DIR, f"{basename}_transcript.json")

    formatter.save(txt_path,  fmt="txt")
    formatter.save(srt_path,  fmt="srt")
    formatter.save(json_path, fmt="json")

    # Keep plain transcript text for WER step
    transcript_text = formatter.to_plain_transcript()

    print(f"\nDone. Output: {OUTPUT_DIR}")

## 6 · Stage 3 (optional) — WER evaluation

In [ ]:
if METADATA_PATH and os.path.isfile(METADATA_PATH):
    print(f"Loading ground-truth metadata: {METADATA_PATH}")
    gt_map = build_ground_truth_map(load_metadata(METADATA_PATH))
    print(f"  {len(gt_map)} ground-truth entries loaded.")

    transcriptions = {basename: transcript_text}
    wer_stats = compute_wer_stats([INPUT_FILE], transcriptions, gt_map)

    print(f"\n── WER Results ──────────────────────────")
    print(f"  Files matched : {wer_stats['count']}")
    if wer_stats["count"] > 0:
        print(f"  Mean WER      : {wer_stats['mean']*100:.2f}%")
        print(f"  Std           : ±{wer_stats['std']*100:.2f}%")
        print(f"  Min / Max     : {wer_stats['min']*100:.2f}% / {wer_stats['max']*100:.2f}%")
        print()
        for pf in wer_stats["per_file"]:
            wer_str = f"{pf['wer']*100:.2f}%" if pf["wer"] is not None else "N/A"
            print(f"  {pf['name']:<40} WER={wer_str}  ref={pf['ref_words']}w  hyp={pf['hyp_words']}w")
else:
    print("METADATA_PATH not set or file not found — skipping WER evaluation.")

## 7 · View transcript

In [ ]:
# Timestamped transcript
with open(txt_path, encoding="utf-8") as f:
    print(f.read())

In [ ]:
# Plain transcript (merged consecutive turns — best for LLM summarisation)
print(formatter.to_plain_transcript())

In [ ]:
# List all output files
for root, dirs, files in os.walk(OUTPUT_DIR):
    for name in sorted(files):
        path = os.path.join(root, name)
        size = os.path.getsize(path)
        print(f"{path}  ({size/1024:.1f} KB)")